# Lesson 1.3 — AWS CLI & boto3 Automation Fundamentals

**Level: Beginner**

## Learning Objectives

By the end of this lesson, you will be able to:

- Configure the AWS CLI with named profiles and MFA
- Use the AWS CLI to query and filter service output with `--query` and `--output`
- Write Python automation scripts using boto3 (sessions, clients, resources, paginators)
- Apply error handling and retry patterns in boto3

**Exam relevance:** SOA-C03 Domain 3 — candidates must perform operations using the AWS Management Console and the AWS CLI.

_Reference: [AWS CLI User Guide](https://docs.aws.amazon.com/cli/latest/userguide/) | [boto3 Documentation](https://boto3.amazonaws.com/v1/documentation/api/latest/index.html)_

## 1. AWS CLI — Configuration

The AWS CLI reads credentials from:
1. Environment variables (`AWS_ACCESS_KEY_ID`, `AWS_SECRET_ACCESS_KEY`)
2. `~/.aws/credentials` file (configured via `aws configure`)
3. IAM roles (for EC2, Lambda, etc.)
4. AWS IAM Identity Center (SSO)

### Named Profiles
You can manage multiple AWS accounts using **named profiles**:
```bash
aws configure --profile prod
aws s3 ls --profile prod
```

### CLI Output Formats
- `--output json` (default) — machine-readable
- `--output table` — human-readable tables
- `--output text` — plain text, good for scripting
- `--output yaml` — YAML format

### JMESPath Queries with `--query`
The `--query` flag uses **JMESPath** syntax to filter JSON output:
```bash
# Get only instance IDs from describe-instances
aws ec2 describe-instances --query 'Reservations[*].Instances[*].InstanceId' --output text

# Get running instances with name tag
aws ec2 describe-instances \
  --filters 'Name=instance-state-name,Values=running' \
  --query 'Reservations[*].Instances[*].{ID:InstanceId,Name:Tags[?Key==`Name`].Value|[0]}' \
  --output table
```

In [ ]:
# Verify CLI setup and run basic commands
import subprocess

def cli(cmd, desc=""):
    """Run an AWS CLI command and print output."""
    if desc:
        print(f"\n=== {desc} ===")
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        print(f"Error: {result.stderr.strip()}")
    else:
        print(result.stdout.strip())
    return result.stdout

cli("aws --version", "AWS CLI Version")
cli("aws sts get-caller-identity", "Current Identity")
cli("aws configure list", "Current Configuration")

In [ ]:
# Practice: AWS CLI with --query and --output

# List S3 buckets (names only)
cli("aws s3api list-buckets --query 'Buckets[*].Name' --output text", "S3 Bucket Names")

# List available EC2 instance types in us-east-1 (first 10)
cli("""
aws ec2 describe-instance-types \
  --region us-east-1 \
  --filters 'Name=current-generation,Values=true' \
  --query 'InstanceTypes[?contains(InstanceType, `t3`)].{Type:InstanceType,vCPU:VCpuInfo.DefaultVCpus,RAM:MemoryInfo.SizeInMiB}' \
  --output table
""", "t3 Instance Types")

## 2. boto3 — AWS SDK for Python

boto3 provides two levels of abstraction:

| Level | Interface | Use Case |
|-------|-----------|----------|
| **Client** | Low-level, 1:1 with API | Full control, all service actions |
| **Resource** | High-level, object-oriented | Simpler code for supported services (S3, EC2, DynamoDB) |

### boto3 Sessions
A **Session** holds credentials and configuration:
```python
import boto3

# Default session (uses ~/.aws/credentials)
session = boto3.Session()

# Named profile session
session = boto3.Session(profile_name='prod')

# Specific region
session = boto3.Session(region_name='eu-west-1')

# Create a client from the session
s3 = session.client('s3')
```

### Paginators
Many AWS APIs return paginated results. Always use **paginators** to get all results:
```python
# WRONG — only returns first page (max 1000 results)
response = s3.list_objects_v2(Bucket='my-bucket')

# CORRECT — handles all pages automatically
paginator = s3.get_paginator('list_objects_v2')
for page in paginator.paginate(Bucket='my-bucket'):
    for obj in page.get('Contents', []):
        print(obj['Key'])
```

In [ ]:
import boto3
from botocore.exceptions import ClientError, BotoCoreError

# boto3 client vs resource demonstration

# CLIENT approach (low-level)
s3_client = boto3.client('s3')
response = s3_client.list_buckets()
buckets_client = [b['Name'] for b in response.get('Buckets', [])]

# RESOURCE approach (high-level)
s3_resource = boto3.resource('s3')
buckets_resource = [b.name for b in s3_resource.buckets.all()]

print("S3 Buckets (via client):", buckets_client)
print("S3 Buckets (via resource):", buckets_resource)
print(f"\nTotal buckets: {len(buckets_client)}")

In [ ]:
# Using Paginators — essential for large-scale CloudOps automation
import boto3

ec2 = boto3.client('ec2', region_name='us-east-1')

# Use paginator to list ALL EC2 instances (handles pagination automatically)
paginator = ec2.get_paginator('describe_instances')

all_instances = []
for page in paginator.paginate(
    Filters=[{'Name': 'instance-state-name', 'Values': ['running', 'stopped']}]
):
    for reservation in page['Reservations']:
        for instance in reservation['Instances']:
            # Extract Name tag safely
            name = next(
                (tag['Value'] for tag in instance.get('Tags', []) if tag['Key'] == 'Name'),
                'No Name'
            )
            all_instances.append({
                'id': instance['InstanceId'],
                'type': instance['InstanceType'],
                'state': instance['State']['Name'],
                'name': name,
                'az': instance['Placement']['AvailabilityZone']
            })

if all_instances:
    print(f"Found {len(all_instances)} instances:")
    for inst in all_instances:
        print(f"  {inst['id']}  {inst['type']:15} {inst['state']:10} {inst['az']:20} {inst['name']}")
else:
    print("No EC2 instances found in us-east-1 (that's fine for a new account).")

In [ ]:
# Error Handling in boto3 — critical for production automation
from botocore.exceptions import ClientError
import boto3

def get_s3_bucket_region(bucket_name: str) -> str:
    """Safely get the region of an S3 bucket with proper error handling."""
    s3 = boto3.client('s3')
    try:
        response = s3.get_bucket_location(Bucket=bucket_name)
        # us-east-1 returns None for LocationConstraint (AWS quirk!)
        region = response['LocationConstraint'] or 'us-east-1'
        return region
    except ClientError as e:
        error_code = e.response['Error']['Code']
        if error_code == 'NoSuchBucket':
            raise ValueError(f"Bucket '{bucket_name}' does not exist.") from e
        elif error_code == 'AccessDenied':
            raise PermissionError(f"Access denied to bucket '{bucket_name}'.") from e
        else:
            raise  # Re-raise unexpected errors

# Test with a bucket that doesn't exist
try:
    region = get_s3_bucket_region('this-bucket-definitely-does-not-exist-12345')
    print(f"Region: {region}")
except ValueError as e:
    print(f"Caught expected error: {e}")
except PermissionError as e:
    print(f"Permission error: {e}")

In [ ]:
# boto3 Waiters — poll until a resource reaches desired state
# Useful for automation: wait for EC2 to be running, wait for RDS to be available

import boto3

ec2 = boto3.client('ec2', region_name='us-east-1')

# List available EC2 waiters
print("Available EC2 Waiters:")
for waiter in ec2.waiter_names:
    print(f"  {waiter}")

# Example usage (don't run unless you have an actual instance launching):
# waiter = ec2.get_waiter('instance_running')
# waiter.wait(InstanceIds=['i-1234567890abcdef0'])
# print("Instance is now running!")

## Exercise

In [ ]:
# TODO: Write a function using boto3 that:
# 1. Lists all S3 buckets in the account
# 2. For each bucket, checks if versioning is enabled
# 3. Returns a dict: {bucket_name: versioning_status}
# Hint: use s3.get_bucket_versioning(Bucket=name)['Status'] — returns 'Enabled', 'Suspended', or '' (not set)

import boto3

def audit_s3_versioning() -> dict:
    """Audit S3 bucket versioning status across all buckets."""
    # YOUR CODE HERE
    pass

result = audit_s3_versioning()
if result:
    for bucket, status in result.items():
        print(f"  {bucket:50} {status}")
else:
    print("No buckets found or function not implemented yet.")

In [ ]:
# TODO: Using the AWS CLI (via subprocess), write a command that:
# Lists all IAM users and their password last used date
# Use --query to extract: UserName, PasswordLastUsed
# Output as a table

import subprocess

cmd = """
aws iam list-users \
  --query 'TODO' \
  --output table
"""

# YOUR CODE HERE
result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
print(result.stdout or result.stderr)

## Summary

- **AWS CLI**: Use `--query` (JMESPath) for filtering, `--output` for format control, `--profile` for multi-account
- **boto3 clients**: Low-level, exact API mapping
- **boto3 resources**: High-level OOP interface (S3, EC2, DynamoDB)
- **Paginators**: Always use for listing operations to avoid missing results
- **Waiters**: Use to poll until resources reach desired state
- **Error handling**: Catch `ClientError`, check `e.response['Error']['Code']` for specific errors

## Key Exam Points
- IAM roles on EC2 = no need to store credentials in code
- CLI uses **credential chain** — environment vars take precedence over config files
- `aws sts assume-role` is used for cross-account access

---

**Next:** [Lesson 1.4 — CloudFormation Basics: Infrastructure as Code](lesson-1.4-cloudformation-basics.ipynb)